# 白川郷 雨＆積雪観光ナビ（予報＆過去1年アーカイブ）

* **氏名：** 小林宙音　高橋拓真　神郡正稀
* **問い：** これから白川郷へ観光に行く際の雨・積雪予報と必要な装備は何か？また、過去1年間の特定の日付にはどれくらい雨や雪が降っていたか？


---

### 1. 何ができるNotebookか（概要）
世界遺産・白川郷（岐阜県大野郡白川村）の天気データを取得し、観光に必要な服装や装備（傘・レインコート・スノーブーツなど）を自動で判定するツール。「今日から7日先までの天気予報」と「過去1年間の指定日の天気アーカイブ」の2つのモードを切り替えて調べることができる。

### 2. 天気データの出典URL
* **Open-Meteo（Weather Forecast API / Historical Weather API）：** https://open-meteo.com

### 3. 自分たちで追加した機能や特徴
* **2つの検索モード切り替え機能：** `input()` を用いて、これからの観光計画に役立つ「7日間予報（モード1）」と、昨年の同日などの傾向を調べられる「過去1年アーカイブ検索（モード2）」を1つのプログラムで選べるようにした。
* **雨量・降雪量の視覚化（ゲージ表示）：** 数値だけでなく `■` の数で雨量（mm）や降雪量（cm）を棒グラフのように表示し、直感的に天候の激しさがわかる工夫をした。
* **具体的な観光装備の自動アドバイス：** 降水量や積雪量のしきい値（10mm以上・10cm以上など）に応じて、「折りたたみ傘」「長傘＋レインコート」「スノーブーツ＆厚手ダウン」など具体的な持ち物を提案するようにした。

### 4. これから改善したいこと、追加したい機能
* **複数観光地への対応：** 現在は白川郷の緯度・経度で固定していますが、周辺の観光地（高山、金沢、富山など）も選択肢から選べるように拡張したい。
* **グラフ描画による可視化：** 現在は文字（`■`）で簡易的なグラフを表示していますが、`matplotlib` などのライブラリを使って気温や降水量の推移を折れ線・棒グラフで出力できるようにしたい。
* **入力エラー対策の強化：** 過去検索で存在しない日付や不正な形式が入力された際に、再入力を促すループ処理を追加してより使いやすくしたい。


In [3]:
pip install requests

In [5]:
import json
from datetime import date, timedelta
from urllib.parse import urlencode
from urllib.request import urlopen

print("==================================================")
print(" 🏔️ 白川郷 雨＆積雪観光ナビ")
print("==================================================")
print("1：これからの天気予報を見る（今日〜7日先）")
print("2：過去1年間の天気から日付を指定して調べる")
mode = input("番号を入力してください（1 または 2）：")

# --------------------------------------------------
# モード1：これからの天気予報（7日分）
# --------------------------------------------------
if mode == "1":
    params = {
        "latitude": 36.2567,
        "longitude": 136.9066,
        "forecast_days": 7,
        "daily": "temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum",
        "timezone": "Asia/Tokyo"
    }
    url = "https://api.open-meteo.com/v1/forecast?" + urlencode(params)

    with urlopen(url) as res:
        weather = json.load(res)

    daily = weather["daily"]
    dates = daily["time"]
    temp_max = daily["temperature_2m_max"]
    temp_min = daily["temperature_2m_min"]
    rain_sum = daily["rain_sum"]
    snow_sum = daily["snowfall_sum"]

    print("\n=== 📅 白川郷 7日間・雨＆積雪予報 ===")
    for i in range(len(dates)):
        d = dates[i]
        tmax = temp_max[i]
        tmin = temp_min[i]
        rain = rain_sum[i]
        snow = snow_sum[i]

        rain_bar = "■" * int(rain)
        snow_bar = "■" * int(snow)

        if snow >= 10.0:
            advice = "⛄ 大雪注意！スノーブーツ＆厚手ダウン必須"
        elif snow > 0:
            advice = "❄️ 雪予報！防水靴と防寒具＋傘を準備"
        elif rain >= 10.0:
            advice = "☔ 本降りの雨！長傘＋レインコート必須"
        elif rain > 0:
            advice = "🌂 小雨予報！折りたたみ傘があると安心"
        else:
            advice = "☀️ 雨も雪もなし！観光日和！"

        print(f"{d} | 最高{tmax:4.1f}℃ / 最低{tmin:4.1f}℃ | 雨:{rain:4.1f}mm {rain_bar} | 雪:{snow:4.1f}cm {snow_bar}")
        print(f"           👉 {advice}")

# --------------------------------------------------
# モード2：過去1年間の日付検索
# --------------------------------------------------
elif mode == "2":
    yesterday = date.today() - timedelta(days=1)
    one_year_ago = date.today() - timedelta(days=365)

    print(f"\n📅 調べられる期間：{one_year_ago} 〜 {yesterday}")
    target_date = input(f"調べたい日付を入力してください（例: {yesterday}）：")

    params = {
        "latitude": 36.2567,
        "longitude": 136.9066,
        "start_date": str(one_year_ago),
        "end_date": str(yesterday),
        "daily": "temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum",
        "timezone": "Asia/Tokyo"
    }
    url = "https://archive-api.open-meteo.com/v1/archive?" + urlencode(params)

    with urlopen(url) as res:
        weather = json.load(res)

    daily = weather["daily"]
    dates = daily["time"]
    temp_max = daily["temperature_2m_max"]
    temp_min = daily["temperature_2m_min"]
    rain_sum = daily["rain_sum"]
    snow_sum = daily["snowfall_sum"]

    found = False
    for i in range(len(dates)):
        if dates[i] == target_date:
            found = True
            tmax = temp_max[i]
            tmin = temp_min[i]
            rain = rain_sum[i]
            snow = snow_sum[i]

            rain_bar = "■" * int(rain)
            snow_bar = "■" * int(snow)

            if snow >= 10.0:
                advice = "⛄ 大雪の日！スノーブーツ・厚手ダウン・長傘が必須でした！"
            elif snow > 0:
                advice = "❄️ 雪が降った日！滑りにくい防水靴と防寒具＋傘が必要でした。"
            elif rain >= 10.0:
                advice = "☔ 本降りの雨の日！長傘＋レインコート必須でした。"
            elif rain > 0:
                advice = "🌂 小雨の日！折りたたみ傘があると安心な天気でした。"
            else:
                advice = "☀️ 雨も雪もなし！傘不要の観光日和でした！"

            print("\n--------------------------------------------------")
            print(f"📍 白川郷 お天気アーカイブ（{target_date}）")
            print(f"🌡️ 最高気温：{tmax} ℃ / 最低気温：{tmin} ℃")
            print(f"💧 雨量  ：{rain:5.1f} mm {rain_bar}")
            print(f"❄️ 降雪量 ：{snow:5.1f} cm {snow_bar}")
            print(f"👉 装備判定：{advice}")
            print("--------------------------------------------------")
            break

    if not found:
        print("\n⚠️ その日付のデータが見つかりませんでした。期間内の日付を入力してください。")

else:
    print("\n⚠️ 1 または 2 を入力してください。")

 🏔️ 白川郷 雨＆積雪観光ナビ
1：これからの天気予報を見る（今日〜7日先）
2：過去1年間の天気から日付を指定して調べる
番号を入力してください（1 または 2）：2

📅 調べられる期間：2025-11-18 〜 2026-11-17
調べたい日付を入力してください（例: 2026-11-17）：2026-11-11

--------------------------------------------------
📍 白川郷 お天気アーカイブ（2026-11-11）
🌡️ 最高気温：13.5 ℃ / 最低気温：1.4 ℃
💧 雨量  ：  0.0 mm 
❄️ 降雪量 ：  0.0 cm 
👉 装備判定：☀️ 雨も雪もなし！傘不要の観光日和でした！
--------------------------------------------------
